# 13 pamoka - Agentų atmintis


## Diegimas

Šiame užrašų knygoje demonstruojama, kaip sukurti kelionių užsakymo agentą su **atsparia atmintimi**, naudojant **Microsoft Agent Framework** (MAF).

Jūs sužinosite, kaip įvairių tipų agento atmintis — darbo, trumpalaikė ir ilgalaikė — veikia tai, kaip agentas išlaiko ir naudoja informaciją pokalbių metu.

**Reikalavimai:**
- Azure AI Foundry projektas su išdiegta pokalbių modeliu (pvz., `gpt-4o-mini`).
- Prisijungta per Azure CLI — terminale paleiskite `az login`.
- `AZURE_AI_PROJECT_ENDPOINT` — jūsų Azure AI Foundry projekto galinis taškas.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — išdiegto modelio pavadinimas.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agentų atminties tipai

DI agentai gali naudoti skirtingų rūšių atmintį, kuri kiekviena atlieka skirtingą paskirtį:

### Darbinė atmintis
Pačios pokalbio gijos — pranešimai, keičiami vienos sesijos metu. Agentas gali grįžti prie ankstesnių pranešimų tame pačiame pokalbyje, kad palaikytų nuoseklumą. MAF tai sukuriama per **`agent.create_session()`**, kuri grąžina `AgentSession`.

### Trumpalaikė atmintis
Informacija, kuri išlieka užduoties ar sesijos metu, bet nėra saugoma visam laikui. Pavyzdžiui, agentas gali kaupti faktus kelių etapų planavimo pokalbio metu ir naudoti juos galutiniam maršrutui sudaryti.

### Ilgalaikė atmintis
Pageidavimai ir faktai, kurie išlieka **per kelias sesijas**. Grįžtantis vartotojas neturėtų kartoti savo mitybos apribojimų ar kelionės stiliaus. Ilgalaikė atmintis paprastai yra paremta išoriniu saugykla — duomenų baze, failu arba vektoriniu indeksu — ir pateikiama agentui per įrankius.


## Darbo atmintis su seansais

Paprastesnė atminties forma yra pokalbio seansas. Kai perduodate tą patį seansą (sukurtą per `agent.create_session()`) einamiesiems `agent.run()` kvietimams, agentas mato visą to pokalbio istoriją ir gali prisiminti ankstesnes detales.

Sukurkime kelionių agentą ir parodykime darbo atmintį.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agentas teisingai prisiminė biudžetą, nes abu pranešimai yra toje pačioje sesijoje. Tai yra **darbinė atmintis** — ji egzistuoja tik sesijos metu.

### Kas nutinka su nauju pokalbiu?

Jei sukuriame **naują** sesiją, agentas neprisimena ankstesnio pokalbio:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Ilgalaikės atminties modelis

Kad prisimintume vartotojo nuostatas **per kelias sesijas**, mums reikia nuolatinės saugyklos, kuri egzistuotų už pokalbio srauto ribų. Agentas prieina prie šios saugyklos naudodamasis **įrankiais** — funkcijomis, kurias gali kviesti informacijos išsaugojimui ir gavimui.

Žemiau įgyvendiname paprastą atmintinę nuostatų saugyklą (realiame naudojime ją palaikytų duomenų bazė arba vektorinė indeksacija) ir pateikiame ją kaip įrankius, kuriuos agentas gali naudoti.

### Architektūra
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Naudotojas pirmą kartą užsisako kelionę jubiliejui

Sarah lankosi pirmą kartą. Agentas turėtų įrašyti jos pageidavimus per įrankius ir naudoti juos viešbučiams rekomenduoti.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 2 scenarijus — Sarah grįžta po kelių savaičių

Sarah pradeda **visiškai naują pokalbį** (simuliuojant naują sesiją). Darbinė atmintis yra tuščia, tačiau ilgalaikė pageidavimų saugykla vis dar turi jos informaciją. Agentas turėtų ją gauti ir panaudoti rekomendacijų suasmeninimui.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Santrauka

Šiame pamokoje sužinojote apie tris agento atminties tipus ir kaip juos įgyvendinti naudojant Microsoft Agent Framework:

| Atminties tipas | MAF mechanizmas | Gyvavimo trukmė |
|---|---|---|
| **Darbinė** | `agent.create_session()` | Viena pokalbio sesija |
| **Trumpalaikė** | Klauso konteksto sukaupiama gijoje | Vienas uždavinys / sesija |
| **Ilgalaikė** | Išorinė saugykla, pasiekiama per `@tool` funkcijas | Tarp sesijų |

### Pagrindinės išvados
1. **`agent.create_session()`** suteikia darbinę atmintį — agentas mato visą pokalbio istoriją sesijos metu.
2. **Naujos sesijos praranda kontekstą** — be ilgalaikės atminties agentas negali prisiminti ankstesnių pokalbių.
3. **`@tool` funkcijos** užpildo spragą — leidžia agentui išsaugoti ir atkurti informaciją iš nuolatinės saugyklos.
4. **Asmeninimas gerėja laikui bėgant** — kuo daugiau pageidavimų saugoma, tuo geresni agento pasiūlymai.

### Praktiniai panaudojimai
- **Klientų aptarnavimas**: prisiminti klientų istoriją ir pageidavimus
- **Asmeniniai padėjėjai**: išlaikyti kontekstą per dienas ar savaites
- **Sveikatos priežiūra**: sekti paciento informaciją ir pageidavimus
- **El. prekyba**: suasmenintas apsipirkimas pagal istoriją

### Tolimesni žingsniai
- Pakeisti atminties žodyną su duomenų baze arba vektoriaus saugykla (pvz., Azure AI Search)
- Pridėti atminties galiojimo laiką laiko jautriai informacijai
- Kurti daugiaveiklio sistemas su bendrą atmintį
- Tyrinėti Cognee užrašų knygutę, skirtą žinių grafų pagrįstai atminčiai


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatizuotuose vertimuose gali būti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Esant svarbiai informacijai rekomenduojame naudotis profesionalaus žmogaus vertimu. Mes neprisiimame atsakomybės už bet kokius nesusipratimus ar klaidingas interpretacijas, kylančias naudojant šį vertimą.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
